# ADMLIV: Own-Price Elasticity Estimation with Simulated Logit Demand

This notebook demonstrates the full ADMLIV workflow for estimating **average own-price elasticities** in a semiparametric demand model. It follows the same data generating process as the Monte Carlo simulations in the paper.

## Workflow Overview

1. **Generate data** from a logit demand model with endogenous pricing
2. **Construct input spaces** using characteristic differences (omega)
3. **Estimate the inverse demand function** using MLIV (KIV or Double Lasso)
4. **Compute plug-in elasticities** (biased due to ML estimation error)
5. **Run ADMLIV** with cross-fitting and PGMM debiasing for valid inference

## Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../replication/demand_model')))

from admliv.utils.featurizers import CoordinatePolyTransform
from admliv.estimators.kiv import KIVEstimator
from admliv.estimators.sieve import DoubleLassoEstimator, DoubleLassoControl, LassoStageControl

from utils.omega_transformer import OmegaTransformer
from utils.raw_data import RawData
from utils.own_price_elasticity import OwnPriceElasticity
from utils.admliv_elasticity import ADMLIVElasticity, ADMLIVElasticityControl
from utils.pgmm_elasticity import PGMMElasticityControl

## 1. Data Generating Process

We simulate a **logit demand model** with $T$ markets and $J$ products per market.

**Utility specification:**
$$\delta_{jt} = \beta_p \cdot p_{jt} + x_{jt}'\beta_x + \xi_{jt}$$

where $\beta_p = -2$ is the price coefficient, $x_{jt}$ are observed characteristics, and $\xi_{jt}$ is an unobserved demand shock.

**Logit market shares:**
$$s_{jt} = \frac{\exp(\delta_{jt})}{1 + \sum_{k=1}^J \exp(\delta_{kt})}$$

**Endogenous pricing:**
$$p_{jt} = \frac{1}{2}\left|1 + \sum_k x_{kjt} + \xi_{jt} + w_{jt} + e_{jt}\right|$$

Price depends on characteristics, the cost shifter $w_{jt}$ (excluded instrument), *and* the demand shock $\xi_{jt}$, creating endogeneity.

**True own-price elasticity:**
$$\varepsilon_{jjt} = \beta_p \cdot p_{jt} \cdot (1 - s_{jt})$$

The parameter of interest is the **average own-price elasticity**: $\theta_0 = \frac{1}{T} \sum_t \varepsilon_{jjt}$

In [ ]:
def simulate_logit_data(n_markets=200, n_products=4, seed=1118):
    """Generate simulated logit demand data with endogenous pricing."""
    np.random.seed(seed)
    
    T, J = n_markets, n_products
    K_1, K_2, K_w = 1, 3, 1
    n = T * J
    market_ids = np.repeat(np.arange(T), J)
    
    # True parameters
    beta_0 = np.array([-2.0, 1.0, -0.5, 0.5, 1.0])
    
    # Draw characteristics, instruments, and shocks
    x_1 = np.random.uniform(0, 1, size=(n, K_1))        # observed (enters utility linearly)
    x_2 = np.random.uniform(0, 1, size=(n, K_2))        # observed (enters nonparametrically)
    x = np.c_[x_1, x_2]
    w = np.random.uniform(0, 1, size=(n, K_w))           # cost shifter (excluded instrument)
    xi = np.random.normal(1, 0.15, size=(n, 1))          # demand shock (unobserved)
    e = np.random.uniform(0, 0.1, size=(n, 1))           # pricing noise
    
    # Endogenous price: depends on x, w, AND xi (source of endogeneity)
    price = 0.5 * np.abs(1 + x.sum(1).reshape(n, 1) + xi + w + e)
    
    # Mean utility and logit shares
    delta = (np.c_[price, x] @ beta_0)[:, np.newaxis] + xi
    delta_mat = delta.reshape(T, J)
    delta_max = delta_mat.max(axis=1, keepdims=True)
    exp_delta_mat = np.exp(delta_mat - delta_max)
    denom = np.exp(-delta_max) + exp_delta_mat.sum(axis=1, keepdims=True)
    shares_mat = exp_delta_mat / denom
    shares = shares_mat.flatten()
    
    # Outside good share and dependent variable (Berry inversion)
    s0 = np.zeros(n)
    for t in range(T):
        mask = market_ids == t
        s0[mask] = 1 - shares[mask].sum()
    y = (np.log(shares) - np.log(s0) - x_1.flatten())
    
    # True average own-price elasticities
    price_mat = price.reshape(T, J)
    elasticities = {}
    for j in range(J):
        eps_j = beta_0[0] * price_mat[:, j] * (1 - shares_mat[:, j])
        elasticities[j] = float(np.mean(eps_j))
    
    return {
        'price': price, 'x1': x_1, 'x2': x_2,
        'shares': shares, 'market_ids': market_ids,
        'y': y, 'xi': xi, 'w': w,
        'elasticities': elasticities,
        'beta_0': beta_0
    }

In [ ]:
data = simulate_logit_data(n_markets=200, n_products=4, seed=1118)

print(f"Markets: {len(np.unique(data['market_ids']))}, Products: {data['x2'].shape[0] // len(np.unique(data['market_ids']))}")
print(f"Total observations: {data['x2'].shape[0]}")
print(f"\nTrue average own-price elasticities:")
for j, eps in data['elasticities'].items():
    print(f"  Product {j}: {eps:.4f}")

### Data Exploration

Price is correlated with the demand shock $\xi$ (endogeneity), but the cost shifter $w$ is excluded from utility.

In [ ]:
data_df = pd.DataFrame({
    'price': data['price'].flatten(),
    'shares': data['shares'],
    'x1': data['x1'].flatten(),
    'w': data['w'].flatten(),
    'xi': data['xi'].flatten()
})
data_df.corr().round(3)

## 2. Construct Input Spaces (Omega Transformations)

In the semiparametric demand model, the inverse demand is a function of **characteristic differences** $\omega_{jt}$ between product $j$ and its rivals in market $t$:

$$\log s_{jt} - \log s_{0t} = x_{1,jt} + \gamma(\omega_{jt}) + \xi_{jt}$$

We construct two omega spaces:
- **Omega** (endogenous): includes price differences, characteristic differences, and shares
- **Omega_IV** (instruments): includes only exogenous characteristic and instrument differences

In [ ]:
# Endogenous omega: includes prices and shares
omega_transformer = OmegaTransformer(
    price_in_diffs=True,
    include_prices=True,
    include_shares=True,
    share_representation='all'
)

# IV omega: only exogenous variables
omega_iv_transformer = OmegaTransformer(
    include_prices=False,
    include_shares=False
)

# Transform
omega = omega_transformer.fit_transform(
    x=data['x2'], market_ids=data['market_ids'],
    price=data['price'], shares=data['shares']
)
x_w = np.c_[data['x1'], data['x2'], data['w']]
omega_iv = omega_iv_transformer.fit_transform(
    x=x_w, market_ids=data['market_ids']
)

print(f"Omega shape: {omega.shape} (endogenous, includes prices + shares)")
print(f"Omega_IV shape: {omega_iv.shape} (instruments, exogenous only)")

## 3. First-Stage: Estimate the Inverse Demand Function

We estimate $\hat{\gamma}(\omega)$ using **Kernel Instrumental Variable (KIV) Estimator**, which regresses the inverted mean utility on omega using omega_IV as instruments.

The dependent variable is $y_{jt} = \log s_{jt} - \log s_{0t} - x_{1,jt}$, the residual after removing the linear component.

In [ ]:
kiv = KIVEstimator(bandwidth_method='median', bandwidth_scale=25.0)
kiv.fit({'Y': data['y'], 'X': omega, 'Z': omega_iv})

# Quick check: predicted vs actual
y_hat = kiv.predict(omega)
r2 = 1 - np.var(data['y'] - y_hat) / np.var(data['y'])
print(f"KIV R-squared: {r2:.4f}")

## 4. Plug-in Elasticity Estimate

The plug-in estimator directly substitutes $\hat{\gamma}$ into the elasticity moment function and computes the sample average:
$$\hat{\theta}_{\text{plugin}} = \frac{1}{T} \sum_t m(W_t; \hat{\gamma})$$

This is **biased** because ML estimation error in $\hat{\gamma}$ introduces a first-order bias term that does not vanish at the $\sqrt{n}$ rate. See Section 8.2 of the paper for the derivation of the elasticity moment function $m$ via the implicit function theorem.

In [ ]:
product_id = 0
raw_data = RawData(
    price=data['price'], x1=data['x1'], x2=data['x2'],
    shares=data['shares'], market_ids=data['market_ids'], w=data['w']
)

moment = OwnPriceElasticity(omega_transformer, product_id=product_id)
plugin_elasticities = moment.compute(kiv, raw_data)

theta_plugin = plugin_elasticities.mean()
theta_true = data['elasticities'][product_id]

print(f"Product {product_id}:")
print(f"  True average elasticity: {theta_true:.4f}")
print(f"  Plug-in estimate:        {theta_plugin:.4f}")
print(f"  Bias:                    {theta_plugin - theta_true:.4f}")

## 5. ADMLIV: Debiased Elasticity Estimation

`ADMLIVElasticity` performs automatic debiased estimation with valid inference:

1. **Cross-fitting**: Splits markets into $K$ folds. For each test fold, trains $\hat{\gamma}$ on the remaining folds.
2. **Double cross-fitting**: Since the elasticity functional is nonlinear in $\gamma$, the Gateaux derivative $M$ depends on $\gamma$. An additional layer of cross-fitting ensures $M$ is computed on held-out data.
3. **PGMM**: Estimates the Riesz representer $\hat{\alpha}(Z)$ via penalized GMM, which provides the debiasing correction.
4. **Debiased estimator**:
$$\hat{\theta} = \frac{1}{n} \sum_i \left[ m(W_i; \hat{\gamma}_{-\ell(i)}) + \hat{\alpha}(Z_i)(Y_i - \hat{\gamma}_{-\ell(i)}(X_i)) \right]$$

The second term corrects the first-order bias, yielding $\sqrt{n}$-consistent and asymptotically normal estimates.

In [ ]:
# ADMLIV with KIV first-stage
control = ADMLIVElasticityControl(
    n_folds=5,
    random_state=42,
    use_adaptive_pgmm=True,
    pgmm_control=PGMMElasticityControl(c=1e-7),
    verbose=True
)

admliv = ADMLIVElasticity(
    mliv_estimator=KIVEstimator(bandwidth_method='median', bandwidth_scale=25.0),
    omega_transformer=OmegaTransformer(
        price_in_diffs=True, include_prices=True,
        include_shares=True, share_representation='all'
    ),
    omega_iv_transformer=OmegaTransformer(
        include_prices=False, include_shares=False
    ),
    omega_featurizer=CoordinatePolyTransform(
        degree=2, pairwise_interactions=True, include_bias=True
    ),
    omega_iv_featurizer=CoordinatePolyTransform(
        degree=2, pairwise_interactions=False, include_bias=True
    ),
    control=control
)

admliv.fit(raw_data, product_ids=[product_id])
result = admliv.results_[product_id]

## 6. Results

In [ ]:
theta_true = data['elasticities'][product_id]

print(f"Product {product_id} — Average Own-Price Elasticity")
print("=" * 55)
print(f"  True value:      {theta_true:.4f}")
print(f"")
print(f"  Plug-in:         {result.theta_plugin:.4f}  (SE = {result.se_plugin:.4f})")
print(f"  Debiased:        {result.theta_debiased:.4f}  (SE = {result.se_debiased:.4f})")
print(f"  95% CI:          [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")
print(f"")
print(f"  Bias correction: {result.theta_debiased - result.theta_plugin:+.4f}")
covers = result.ci_lower <= theta_true <= result.ci_upper
print(f"  CI covers true:  {covers}")

## 7. Alternative First-Stage: Double Lasso

ADMLIV is agnostic to the first-stage MLIV estimator. Here we swap in Double Lasso, which uses LASSO in both stages for high-dimensional IV estimation.

In [ ]:
# Double Lasso first-stage
dl_control = DoubleLassoControl.with_fixed_alpha(
    fs_alpha=1e-4, ss_alpha=1e-8, max_iter=10000
)
dl_estimator = DoubleLassoEstimator(
    x_featurizer=CoordinatePolyTransform(degree=2, pairwise_interactions=True, include_bias=False),
    z_featurizer=CoordinatePolyTransform(degree=3, pairwise_interactions=True, include_bias=False),
    control=dl_control
)

control_dl = ADMLIVElasticityControl(
    n_folds=5,
    random_state=42,
    use_adaptive_pgmm=True,
    pgmm_control=PGMMElasticityControl(c=1e-7),
    verbose=False
)

admliv_dl = ADMLIVElasticity(
    mliv_estimator=dl_estimator,
    omega_transformer=OmegaTransformer(
        price_in_diffs=True, include_prices=True,
        include_shares=True, share_representation='all'
    ),
    omega_iv_transformer=OmegaTransformer(
        include_prices=False, include_shares=False
    ),
    omega_featurizer=CoordinatePolyTransform(
        degree=2, pairwise_interactions=True, include_bias=True
    ),
    omega_iv_featurizer=CoordinatePolyTransform(
        degree=2, pairwise_interactions=False, include_bias=True
    ),
    control=control_dl
)

admliv_dl.fit(raw_data, product_ids=[product_id])
result_dl = admliv_dl.results_[product_id]

print(f"Product {product_id} — Double Lasso First-Stage")
print("=" * 55)
print(f"  True value:      {theta_true:.4f}")
print(f"  Plug-in:         {result_dl.theta_plugin:.4f}  (SE = {result_dl.se_plugin:.4f})")
print(f"  Debiased:        {result_dl.theta_debiased:.4f}  (SE = {result_dl.se_debiased:.4f})")
print(f"  95% CI:          [{result_dl.ci_lower:.4f}, {result_dl.ci_upper:.4f}]")
covers_dl = result_dl.ci_lower <= theta_true <= result_dl.ci_upper
print(f"  CI covers true:  {covers_dl}")

## 8. Comparison

In [ ]:
comparison = pd.DataFrame({
    'Method': ['KIV + ADMLIV', 'Double Lasso + ADMLIV'],
    'Plugin': [result.theta_plugin, result_dl.theta_plugin],
    'Debiased': [result.theta_debiased, result_dl.theta_debiased],
    'SE': [result.se_debiased, result_dl.se_debiased],
    'CI Lower': [result.ci_lower, result_dl.ci_lower],
    'CI Upper': [result.ci_upper, result_dl.ci_upper],
}).set_index('Method')

print(f"True average elasticity (product {product_id}): {theta_true:.4f}\n")
comparison.round(4)